# TensorRT engine build + benchmark (Colab GPU)
Portable ONNX → GPU-specific FP16 engine, with numbers.
Proves in order: (1) GPU present, (2) FP32 engine builds, (3) FP16 engine builds via AutoCast,
(4) FP16 ≈ ONNX end-to-end on same pixels, (5) latency of 4 backends, GPU-synced.
Inputs: `model.onnx` + `model.onnx.data` (external weights: BOTH required) + `manifest.json`.
Honesty rules: report GPU model + TRT version next to every number. Engines are NOT
portable and never committed (`*.engine` → GitHub Release asset).

TRT 11 notes baked in: no `trtexec` in the pip wheel (we build via `trt.Builder`),
`EXPLICIT_BATCH`/`BuilderFlag.FP16` removed (strong typing: precision lives in the ONNX graph),
`parse_from_file` required for external weights, `Logger(WARNING)` enum-only.

## 0 — GPU check (fail fast)
No NVIDIA GPU → stop. TRT builds a program for one GPU arch; on CPU there is nothing to build.

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert gpu, "no NVIDIA GPU"
print("GPU:", gpu)

## 1 — Install (TRT 11 + torch baseline + ModelOpt for FP16)
`pip install tensorrt` pulls CUDA/cuDNN wheels. If `import tensorrt` segfaults: restart runtime, re-run.

In [ ]:
!pip install -q tensorrt polygraphy onnxruntime-gpu pillow numpy onnx 2>&1 | tail -2
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -1
!pip install -q transformers "nvidia-modelopt[onnx]" 2>&1 | tail -1
import tensorrt as trt
print("TensorRT:", trt.__version__)
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available()

## 2 — Inputs (same docs image as export_onnx.py, so parity is comparable)
Fresh session: if `artifacts/` is empty the cell opens the Colab uploader —
select `model.onnx` + `model.onnx.data` (BOTH, external weights).

In [ ]:
import io, urllib.request
from pathlib import Path
from PIL import Image
MODEL_ID = "Roboflow/rf-detr-nano"  # single source: build (§3), parity (§6), serving
ART = Path("artifacts")
if not ((ART / "model.onnx").exists() and (ART / "model.onnx.data").exists()):
    try:  # fresh session: pull the two files in via Colab uploader
        from google.colab import files
        print("Upload model.onnx AND model.onnx.data (external weights, both required)")
        up = files.upload()
        ART.mkdir(exist_ok=True)
        for name, data in up.items():
            (ART / Path(name).name).write_bytes(data)
    except ImportError:
        pass
assert (ART / "model.onnx").exists() and (ART / "model.onnx.data").exists(), \
    "model.onnx + model.onnx.data missing: upload both into artifacts/"
SAMPLE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(io.BytesIO(urllib.request.urlopen(SAMPLE_URL).read())).convert("RGB")
print("sample:", image.size, image.mode)

## 3 — FP32 engine via TensorRT Python API
pip tensorrt has NO trtexec binary → build with `trt.Builder`.
`parse_from_file` (not `parse(bytes)`) so the parser resolves `model.onnx.data`.
Successful builds also write `<engine>.meta.json` (gpu + TensorRT + IO spec) —
the serving layer validates it before loading the engine.

In [ ]:
import json, subprocess, tensorrt as trt
from pathlib import Path
def build_engine(onnx_path, engine_path, model_id=MODEL_ID):
    logger = trt.Logger(trt.Logger.VERBOSE)  # VERBOSE shows WHICH op fails
    builder = trt.Builder(logger)
    network = builder.create_network()  # TRT11: explicit-batch is the default
    parser = trt.OnnxParser(network, logger)
    ok = parser.parse_from_file(str(onnx_path))
    if not ok:
        for i in range(parser.num_errors):
            print('PARSE ERROR:', parser.get_error(i))
        raise RuntimeError('ONNX parse failed (details above)')
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 4 << 30)
    print('building... (takes minutes on transformer)')
    eng = builder.build_serialized_network(network, config)
    if eng is None:
        raise RuntimeError('build returned None: unsupported op, see VERBOSE log')
    eng = bytes(eng)  # IHostMemory -> bytes for write_bytes
    Path(engine_path).write_bytes(eng)
    print(f'saved {engine_path} ({len(eng)/1e6:.1f} MB)')
    # Sidecar: TrtDetector refuses an engine whose gpu / TensorRT /
    # IO spec differs from this box — write the proof next to the artifact.
    rt = trt.Runtime(trt.Logger(trt.Logger.WARNING))
    e = rt.deserialize_cuda_engine(eng)
    mode = {n: e.get_tensor_mode(n) for n in e}
    spec = {n: list(e.get_tensor_shape(n)) for n in e}
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                              capture_output=True, text=True).stdout.strip()
    side = {
        'engine': Path(engine_path).name,
        'gpu': gpu_name,
        'tensorrt': trt.__version__,
        'model_id': model_id,
        'precision': 'fp16' if 'fp16' in str(engine_path) else 'fp32',
        'inputs': {n: spec[n] for n in spec if mode[n] == trt.TensorIOMode.INPUT},
        'outputs': {n: spec[n] for n in spec if mode[n] == trt.TensorIOMode.OUTPUT},
    }
    side_path = Path(str(engine_path) + '.meta.json')
    side_path.write_text(json.dumps(side, indent=2))
    print(f'sidecar {side_path.name}: gpu={gpu_name!r} trt={trt.__version__} {side["inputs"]}')
build_engine('artifacts/model.onnx', 'artifacts/model_fp32.engine')
print('FP32 OK')

## 4 — FP16 engine via ModelOpt AutoCast (TRT 11 strong typing)
TRT 11 removed `BuilderFlag.FP16`: precision now lives in the ONNX graph itself.
AutoCast converts compute layers to FP16; `--keep_io_types` keeps I/O FP32 so the feed is unchanged.
Norm/softmax ops + detection head stay FP32 (FP16-unstable). FP32 engine from §3 stays valid.

In [ ]:
# TRT11: precision lives in the ONNX graph (BuilderFlag.FP16 removed) → ModelOpt AutoCast.
# --keep_io_types: I/O stay FP32 so the ORT/TRT feed is unchanged.
# --op_types_to_exclude: LayerNorm/Softmax/ReduceMean are FP16-unstable on transformers.
# --nodes_to_exclude: detection head kept FP32 (logits overflow FP16 max 65504).
!python -m modelopt.onnx.autocast --onnx_path artifacts/model.onnx --output_path artifacts/model_fp16_autocast.onnx --keep_io_types --op_types_to_exclude LayerNormalization Softmax ReduceMean --nodes_to_exclude ".*class.*|.*logit.*|.*bbox.*|.*head.*" 2>&1 | tail -3
assert Path("artifacts/model_fp16_autocast.onnx").exists(), "autocast failed, read output above"
print("autocast OK")
build_engine('artifacts/model_fp16_autocast.onnx', 'artifacts/model_fp16.engine')
print('FP16 OK')

## 5 — Which weights went FP16?
TRT 11 has no per-layer engine query (`get_layer` removed): precision is declared in the
AutoCast ONNX graph. Expect most weights FP16; I/O stay FP32. One line of output.

In [ ]:
import onnx
from collections import Counter
# TRT11 strong typing: precision lives in the ONNX graph, not the engine API (get_layer removed).
# Count initializer dtypes in the AutoCast graph: instant, one line of output.
m = onnx.load("artifacts/model_fp16_autocast.onnx")
c = Counter(t.data_type for t in m.graph.initializer)
names = {1: "FLOAT", 10: "FLOAT16", 7: "INT64 (non-compute constants)"}
print("FP16 weight mix:", {names.get(k, k): v for k, v in c.items()})

## 6 — Parity: TRT vs ONNX Runtime, same pixels
Two gates, both rooted in the export parity contract:
- **FP32 engine**: tensor-level `allclose(rtol=1e-3)` — must be bit-close.
- **FP16 engine**: the graph's internal TopK (300-of-576) flips indices whenever score gaps
  fall below FP16 noise — 549/575 gaps do — so tensor-level parity on `logits` is
  structurally unattainable. Gate END-TO-END instead: same detections above 0.5,
  Δconf ≤ 0.05 and Δbox ≤ 0.01 on those detections (the contract the product sells).
Measured: FP32 max|d|=2.09e-04; FP16 Δconf=0.0053, Δbox=0.0011, 4/4 same detections.

In [ ]:
import numpy as np, onnxruntime as ort, tensorrt as trt
from transformers import AutoImageProcessor
KEY_OUTPUTS = ["logits", "pred_boxes"]
RTOL, ATOL = 1e-3, 1e-4
feed = {"pixel_values": np.asarray(AutoImageProcessor.from_pretrained(MODEL_ID)(images=image, return_tensors="np")["pixel_values"], dtype=np.float32), "pixel_mask": np.asarray(AutoImageProcessor.from_pretrained(MODEL_ID)(images=image, return_tensors="np")["pixel_mask"], dtype=np.int64)}
ort_sess = ort.InferenceSession("artifacts/model.onnx", providers=["CUDAExecutionProvider"])
ref = dict(zip(KEY_OUTPUTS, ort_sess.run(KEY_OUTPUTS, feed)))
print("reference:", {k: v.shape for k, v in ref.items()})

In [ ]:
def trt_infer(engine_path, feed):
    import numpy as np, torch, tensorrt as trt
    _NP2TORCH = {np.dtype('float32'): torch.float32, np.dtype('float16'): torch.float16,
                 np.dtype('int64'): torch.int64, np.dtype('int32'): torch.int32}
    eng = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open(engine_path, 'rb').read())
    ctx = eng.create_execution_context()
    ten = {}
    for n in eng:
        dt = np.dtype(trt.nptype(eng.get_tensor_dtype(n)))
        if eng.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
            t = torch.from_numpy(np.ascontiguousarray(feed[n].astype(dt))).cuda()
        else:
            t = torch.empty(tuple(eng.get_tensor_shape(n)), dtype=_NP2TORCH[dt]).cuda()
        ten[n] = t
        ctx.set_tensor_address(n, t.data_ptr())
    ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
    torch.cuda.synchronize()
    out = {}
    for k in KEY_OUTPUTS:
        t = ten[k]
        out[k] = t.float().cpu().numpy() if t.is_floating_point() else t.cpu().numpy()
    return out
def end2end(ref, got, conf_thr=0.5, dconf_max=0.05, dbox_max=0.01):
    p32, p16 = sigmoid(ref['logits']), sigmoid(got['logits'])
    c32, c16 = p32.max(-1), p16.max(-1)
    d32, d16 = c32 > conf_thr, c16 > conf_thr
    mask = d32 | d16
    dconf = float(np.abs(c32 - c16)[mask].max()) if mask.any() else 0.0
    dbox = float(np.abs(ref['pred_boxes'] - got['pred_boxes'])[mask].max()) if mask.any() else 0.0
    same = bool((d32 == d16).all())
    return same and dconf <= dconf_max and dbox <= dbox_max, dict(same=same, n=int(d32.sum()), dconf=round(dconf, 5), dbox=round(dbox, 5))
sigmoid = lambda x: 1 / (1 + np.exp(-x.astype(np.float64)))
for tag, p in [('FP32', 'artifacts/model_fp32.engine'), ('FP16', 'artifacts/model_fp16.engine')]:
    got = trt_infer(p, feed)
    if tag == 'FP32':  # tensor gate: FP32 must be bit-close
        for k in KEY_OUTPUTS:
            d = float(np.abs(got[k].astype(np.float64) - ref[k].astype(np.float64)).max())
            ok = bool(np.allclose(got[k], ref[k], rtol=RTOL, atol=ATOL))
            print(f'tensor {tag} {k}:', 'OK' if ok else 'FAIL', f'max|d|={d:.2e}')
            assert ok, f'{tag} {k} diverges'
    ok, st = end2end(ref, got)
    print(f'e2e {tag}:', 'OK' if ok else 'FAIL', st)
    assert ok, f'{tag} end-to-end diverges'

## 7 — Benchmark (warmup, GPU-synced, median + p99)
CUDA launches are async: unsynced timing measures queue overhead, not inference.
Four backends: PyTorch FP16, ONNX Runtime CUDA, TRT FP32, TRT FP16.

In [ ]:
import time, torch
from transformers import RfDetrForObjectDetection
WARMUP, TIMED = 10, 50
print("ORT providers:", ort_sess.get_providers())  # must contain CUDAExecutionProvider
if "CUDAExecutionProvider" not in ort_sess.get_providers():
    print("WARNING: CUDAExecutionProvider missing -> ORT row measures CPU, not GPU")
# Engines + execution contexts + device buffers: allocated ONCE, outside the timer.
# Previous versions deserialized the engine per call, then created the context per call;
# both are setup costs, not inference, and inflated every TRT row.
_NP2TORCH = {np.dtype('float32'): torch.float32, np.dtype('float16'): torch.float16,
             np.dtype('int64'): torch.int64, np.dtype('int32'): torch.int32}
def make_runner(path):
    engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open(path, 'rb').read())
    ctx = engine.create_execution_context()
    ten = {}
    for n in engine:
        dt = np.dtype(trt.nptype(engine.get_tensor_dtype(n)))
        t = torch.empty(tuple(engine.get_tensor_shape(n)), dtype=_NP2TORCH[dt], device='cuda')
        ten[n] = t
        ctx.set_tensor_address(n, t.data_ptr())
    def _run(feed):
        for n in engine:
            if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT:
                src = np.ascontiguousarray(feed[n].astype(np.dtype(trt.nptype(engine.get_tensor_dtype(n))), copy=False))
                ten[n].copy_(torch.from_numpy(src))
        ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        torch.cuda.synchronize()
        return {k: ten[k].cpu().numpy() for k in KEY_OUTPUTS}
    return _run
run_fp32 = make_runner("artifacts/model_fp32.engine")
run_fp16 = make_runner("artifacts/model_fp16.engine")
pt = RfDetrForObjectDetection.from_pretrained(MODEL_ID).eval().half().cuda()
ptf = {k: torch.from_numpy(v).cuda() for k, v in feed.items()}
def bench(fn, tag):
    [fn() for _ in range(WARMUP)]
    s = []
    for _ in range(TIMED):
        t0 = time.perf_counter(); fn(); torch.cuda.synchronize(); s.append((time.perf_counter() - t0) * 1000)
    s.sort()
    row = {"backend": tag, "p50_ms": round(s[TIMED // 2], 2), "p99_ms": round(s[int(TIMED * 0.99) - 1], 2)}
    print(f"{tag:24s} p50 {row['p50_ms']:8.2f} ms  p99 {row['p99_ms']:8.2f} ms")
    return row
with torch.inference_mode():
    rows = [bench(lambda: pt(**ptf), "pytorch-fp16-cuda"),
            bench(lambda: ort_sess.run(KEY_OUTPUTS, feed), "onnxruntime-cuda"),
            bench(lambda: run_fp32(feed), "tensorrt-fp32"),
            bench(lambda: run_fp16(feed), "tensorrt-fp16")]
import json, subprocess
from pathlib import Path
rep = {"gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
       "tensorrt": trt.__version__, "model": MODEL_ID,
       "method": f"warmup={WARMUP}, timed={TIMED}, forward-only, gpu-synced, engines+contexts allocated once",
       "results": rows}
Path("artifacts/benchmark.json").write_text(json.dumps(rep, indent=2))
print("saved artifacts/benchmark.json")

## 8 — Bring home
1. Download `artifacts/benchmark.json` (folder icon → ⋮ → Download) → drop it into the
repo `artifacts/` (git re-includes exactly this file). 2. FP16 engine stays on Colab/Drive
(GPU+TRT specific) — engine + `.meta.json` sidecar travel together
(the serving layer refuses one without the other). 3. Paste the 4 p50/p99 rows into `docs/RESULTS.md` with GPU + TRT.
Serving layer (implemented): `TrtDetector` + `create_detector()` read the sidecar written
in §3 and fall back to ONNX — loudly — when engine or metadata is unusable here.